# The nodejax cookbook

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/nodejax/blob/main/docs/cookbook.ipynb)

This cookbook starts from a minimal node and builds it out, explaining as it goes. Every snippet runs as written, and each section uses what the previous one defined.

## 1. A first node

In [ ]:
import jax
import jax.numpy as jnp
import optax
from nodejax import node, Leaf, Struct, Aux, train_step, trained, Composite
from nodejax import batch, ensemble, stack, nn

@node
def Gain():
    def param(scale):
        return Struct(scale=scale)
    def apply(param, input):
        return param.scale * input
    return Leaf(apply, param=param)

model = Gain().parameterize(scale=2.0)
assert model.apply(3.0) == 6.0

A node is at most three pure functions; this one needs two. `param` is the constructor: its signature declares what a caller must supply (`scale`, required because it has no default), and `parameterize` runs it and binds the result. `apply` computes. The names in the signatures are the specification; there is no other registration step. The `@node` decorator records the construction on the node the factory returns (which is what later respecialization re-runs) and names it after the factory, so this node is called `gain` without anyone writing the string.

## 2. State

In [ ]:
@node
def Integrator():
    def param(gain):
        return Struct(gain=gain)
    def init(param):
        return jnp.asarray(0.0)
    def apply(param, state, input):
        new = state + param.gain * input
        return new, new
    return Leaf(apply, param=param, init=init)

model = Integrator().parameterize(gain=1.0).initialize()
model, y = model(5.0)                  # each call returns a successor
final, trajectory = model.scan(jnp.ones(10))
assert trajectory[-1] == 15.0

The third function, `init`, builds the starting state, and its presence makes the node CYCLIC: output feeding back into the next step. Binding proceeds in order: `parameterize` binds params, `initialize` constructs and binds state. Note that `model` is an immutable object; A call to a cyclic bound node returns a successor binding the next state, and never mutates the original binding. `scan` runs it over a time axis, jitted, and hands back the advanced model with the outputs; note it continued from 5, where the single step left off.

## 3. Composition

In [ ]:
pipe = Gain().parameterize(scale=2.0) >> Integrator().parameterize(gain=1.0)
model = pipe.initialize()

final, trajectory = model.scan(jnp.ones(10))
assert trajectory[-1] == 20.0
assert model.param.gain.scale == 2.0     # params: a tree named by member
assert final.state.integrator == 20.0      # state: same shape, same names

`>>` chains nodes into a node, and members may enter already bound: a composition of bound members comes back bound. The composite's params and state are trees keyed by member name (the nodes' own names, captured from the factories), and both remain plain pytrees: read them, `jax.tree.map` them, `jax.grad` through them. The state threading between members that you would otherwise write by hand does not exist as user code.

## 4. Shapes and keys

In [ ]:
@node
def Linear(n_out):
    def param(node, rng):
        n_in = node.input.shape[-1]
        return Struct(w=jax.random.normal(rng.next(), (n_in, n_out)) / jnp.sqrt(n_in),
                      b=jnp.zeros(n_out))
    def apply(param, input):
        return input @ param.w + param.b
    return Leaf(apply, param=param)

relu = Leaf(lambda input: jnp.maximum(input, 0.0), name='relu')

X = jax.random.normal(jax.random.PRNGKey(0), (32, 4))
net = Linear(8) >> relu >> Linear(1)
model = net.with_input(X).parameterize(rng=jax.random.PRNGKey(1))
assert model.param.linear.w.shape == (4, 8)
assert model.apply(X).shape == (32, 1)

Two reserved names appear. `node` hands the constructor its own definition, resolved, so it can read the input shape; `with_input(X)` is where that shape comes from, and the pipe walks it member by member so the second linear sees the first one's output width. `rng` declares a dependency on randomness, and what arrives is a scope-local stream of keys: every `rng.next()` yields a fresh key, as many as the body needs, with no line anywhere dedicated to splitting or naming keys. The flow of keys stays explicit at the function boundary, since the signature says a key is owed and the caller must pass one, while the plumbing below the boundary disappears: the pipe splits the one key so every member's stream is independent.

## 5. Training

In [ ]:
def mse(pred, target):
    return jnp.mean((pred - target) ** 2)

y = X @ jnp.ones((4, 1))
steps = 300
train_x = jnp.broadcast_to(X, (steps, *X.shape))
train_y = jnp.broadcast_to(y, (steps, *y.shape))

trainer = train_step(model.initialize(), mse, optax.adam(1e-2))
fitted, aux = trained(trainer).apply(input=train_x, target=train_y)
assert aux.loss[-1] < 0.01
_, preds = fitted(X)                       # the trained model, ready to call
assert preds.shape == (32, 1)

`train_step` consumes the model as bound as you hand it over, and returns the trainer bound to match. Here the model is fully bound (`initialize` is a trivial step for a stateless net, and spelling it keeps the binding order visible), so the trainer arrives state-bound too, born from the model's bindings: its param is the starting weights, its state is the optimizer primed on them beside the model's own state. One rule spans every binding stage: a binding overrides what a constructor would build, and never replaces the constructor, so the trainer's def still knows how to construct fresh instances of itself. `trained` runs the trainer over the sequence and finalizes: what comes back IS the trained model, state-bound and callable, the optimizer scaffolding struck, the loss trace riding aux. When you want to own the loop instead, the trainer is born ready, no initialize step: a training run is the same scan as section 2.

In [ ]:
trainer, (_, aux) = trainer.scan(input=train_x, target=train_y)
assert aux.loss[-1] < 0.01
assert trainer.state.opt.params.linear.w.shape == (4, 8)   # readable mid-training

The loop around it is your own Python: chunk it, log anything (the trainer is data), stop when you like, keep calling the trainer to resume.

## 6. Randomness and aux at apply time

In [ ]:
@node
def Jitter():
    def param(sigma):
        return Struct(sigma=sigma)
    def apply(param, input, rng):
        return input + param.sigma * jax.random.normal(rng.next())
    return Leaf(apply, param=param)

noisy = Jitter().parameterize(sigma=0.1)
a = noisy.apply(input=1.0, rng=jax.random.PRNGKey(0))
b = noisy.apply(input=1.0, rng=jax.random.PRNGKey(0))
assert a == b                              # same key, same draw: still pure

An apply that draws noise declares the reserved `rng` role and receives a scope-local stream separate from its input bundle. Draw with `rng.next()` as often as the step needs. Purity is preserved: the same supplied key reproduces the same draws. The other automated report is aux: an apply with something secondary to report returns `output, Aux(...)` and carries on. Both automate through composition:

In [ ]:
@node
def Probe():
    def apply(input):
        return input, Aux(norm=jnp.linalg.norm(input))
    return Leaf(apply)

wired = Linear(8) >> Jitter() >> Probe() >> relu >> Linear(1)
model = wired.with_input(X).parameterize(rng=jax.random.PRNGKey(1),
                                         jitter=Struct(sigma=0.1))
out, aux = model.apply(input=X, rng=jax.random.PRNGKey(4))
assert out.shape == (32, 1)                # the wire: aux never touched it
assert aux.probe.norm.shape == ()          # the report, keyed by member

The jitter's key requirement bubbled to the boundary: the caller owes ONE key, and the pipe splits it toward every member that declared the need. The probe's report surfaced at the top under its member name, while the downstream members saw the raw signal. Neither took a line of plumbing, and both stack naturally under the transforms of section 8: scan stacks aux over time, ensemble over members.

## 7. Wiring by hand

In [ ]:
@node
def Highway(width):
    members = Composite(gate=Linear(width),
                        body=Linear(width) >> relu >> Linear(width))

    def apply(self, input):
        mix = jax.nn.sigmoid(self.gate(input))
        return mix * self.body(input) + (1 - mix) * input

    return members(apply)

model = Highway(4).with_input(X).parameterize(rng=jax.random.PRNGKey(2))
assert model.apply(X).shape == (32, 4)

When `>>` is not the shape of your dataflow, declare the structure first: `Composite(**members)` holds the member tree immutably, and calling it with the behavior builds the node. Inside `apply`, `self` is a scope-local mutable object-like view of that structure, bound to the live params and state: calling a member (`self.gate(input)`) runs it and advances its state slice in place, reads see the current values, and you write ordinary imperative wiring. Like the key stream, it is purely a scope-local abstraction: the sugar transforms the function into an ordinary pure `apply`, and to anything outside, only the node contract is visible, which is why the highway composes onward like any node. A composite is for wiring SEVERAL members; wrapping exactly one node is the wrapper shape, and the stock `residual(body)` transform is that: `x + f(x)` around any shape-preserving node, no structure to declare.

## 8. The axis family: batch, ensemble, stack

Because every node declares which tree is params and which is state, a transform can put a mapped axis exactly where it belongs. The family is one trick three ways: batch maps the DATA, ensemble maps the PARAMS, stack turns the param axis into DEPTH.

In [ ]:
per_sample = Linear(4) >> nn.BatchNorm(0.9)
norming = batch(per_sample).with_input(X).parameterize(
    rng=jax.random.PRNGKey(5)).initialize()
assert norming.param.linear.w.shape == (4, 4)       # params shared: no batch axis
assert norming.state.batch_norm.mean.shape == (4,)  # running stats: ONE copy, not 32
norming, out = norming(X)                           # the stats advance as ordinary state
assert out.shape == (32, 4)

`batch` is vmap over data: the pipe stays written per sample, params broadcast, and state maps per element. The batch norm is the interesting tenant: its moments are collectives over the named batch axis, so it drops into the per-sample pipe as one term, and its running stats are tagged as a population quantity, one copy for the whole batch, because that is what a batch statistic IS. No train flag anywhere: the stats are ordinary state, advancing while you call and sitting still when you stop.

In [ ]:
population = ensemble(wired.with_input(X), n=4).parameterize(
    rng=jax.random.PRNGKey(3), jitter=Struct(sigma=0.1))
assert population.param.linear.w.shape == (4, 4, 8)   # a member axis, stacked
out, aux = population.apply(input=X, rng=jax.random.PRNGKey(4))
assert out.shape == (4, 32, 1)             # four models, one call
assert aux.probe.norm.shape == (4,)        # the probe's report grew the axis too

`ensemble` is vmap over params: one definition, `n` independent parameter sets (the one key split per member), every member answering the same input in one call. Section 6's pipe rides through whole: the jitter's key requirement still bubbles to the boundary and now splits per member as well as per draw, and the probe's aux comes back with the member axis on it. A deep ensemble is this plus `train_step`, nothing more.

In [ ]:
deep = stack(Linear(8) >> relu, n=3).with_input(jnp.zeros(8)).parameterize(
    rng=jax.random.PRNGKey(6))
assert deep.param.linear.w.shape == (3, 8, 8)   # a layer axis on the same tree
assert deep.apply(jnp.ones(8)).shape == (8,)    # layer k fed layer k+1

`stack` puts the same axis to a third use: the stacked params are LAYERS, and apply scans over them, layer k feeding layer k+1. One layer definition compiles once whatever the depth, which is what makes hundred-layer towers cheap to build and trace. Each of the three returns a node, so they nest with each other and with `scan` and `train_step`; the README's opening block is nothing but this, stacked deep.

## 9. Statics, specialize, and generics

In [ ]:
wider = net.specialize(**{'linear.n_out': 16})
retrained = wider.with_input(X).parameterize(rng=jax.random.PRNGKey(8))
assert retrained.param.linear.w.shape == (4, 16)
assert retrained.param.linear_2.w.shape == (16, 1)   # downstream re-derived

Statics are the binding stage BEFORE params: values that decide the node's structure (sizes, rates, architectural choices), bound at construction, never entering a pytree. This mirrors the divide jax itself is built on, what decides a trace versus what flows through it. The `@node` decorator records each factory call, the factory and the full argument set it ran with, and that record is what makes statics live values rather than frozen history: `specialize` reconfigures a BUILT tree by re-running the record, `'linear.n_out'` addressing one member, `'*.field'` broadcasting to every leaf that declares the field. The second linear's fan-in re-derived without being mentioned, because the build genuinely re-ran.

In [ ]:
unit = Linear()                                # n_out unbound: a generic
assert unit.generic
tower = stack(unit, n=3)
print(tower.describe())
assert list(tower.statics_by_path()) == ['layer.n_out', 'n']

column = tower.specialize(**{'layer.n_out': 8}).with_input(jnp.zeros(8)).parameterize(
    rng=jax.random.PRNGKey(9))
assert column.param.w.shape == (3, 8, 8)       # unbound, filled through the stack

The same arguments left unbound make a generic: call a factory with statics missing and the product is the same description with unbound static arguments, `node.generic` saying which end of the spectrum it is on. Unbound statics ride through any assembly (this one crossed `stack`), and `specialize` supplies the remainder: one more binding stage, `specialize` to statics what `parameterize` is to params and `initialize` to state, with the later stages refusing to bind on a generic. `.statics_by_path()` lists every recorded argument at its full address; here `layer.n_out` identifies the unbound static under `stack`. `.describe()` writes any tree out, with unbound values marked and bindings tallied. A filled tree behaves identically to one built concrete; `nodejax/tests/test_generic_stress.py` holds that claim under transforms and trainers alike.

## Where next

The [README](../readme.md) for the view from the top, [`docs/handbook.md`](handbook.md) for the patient reference, and the test suite for working examples of everything: [`nodejax/examples/`](../nodejax/examples/) and [`nodejax/tests/`](../nodejax/tests/) are written to be read.